<a href="https://colab.research.google.com/github/ConradKatlegoMogane/Data-Dump-For-all-things-useful-/blob/main/DPSA_Data_Mining_Project_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Exercutive Summary**

The page outlines a Python-based data mining project designed to extract job post information from PDF circulars published by the Department of Public Service and Administration (DPSA) in South Africa. Here's a concise summary of the key components:

- **PDF Handling**: The script uses the PyMuPDF library (`fitz`) to open and read text from a DPSA PDF job circular stored on Google Drive.
- **Text Extraction & Splitting**: It processes the entire PDF, splits it by job post identifiers, and extracts structured information using regular expressions.
- **Captured Fields**:
  - Post title
  - Centre (location)
  - Salary
  - Requirements
  - Duties
  - Enquiries contact
  - Closing date (extracted from the full document rather than per post)
- **Output**: The collected data for each post is stored in a pandas DataFrame and can be saved or analyzed further.
- **Result**: The script reports a total of **313 job positions** extracted from the circular.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 70.2 MB/s eta 0:00:00


This python script works with all pdfs published on [Department of public service](https://https://www.dpsa.gov.za/newsroom/psvc/) making navigation through these job easier

In [ ]:
import fitz  # PyMuPDF
import re
import pandas as pd

# Load the PDF
doc = fitz.open(r"/content/drive/MyDrive/PSV CIRCULAR 07 OF 2025.pdf")# or use the full path if not in the same directory

# Extract all text from the document
text = ""
for page in doc:
    text += page.get_text()

# Close the document
doc.close()

# Split text by each post
posts = re.split(r"\nPOST\s+\d+/\d+\s*:\s*", text)[1:]  # skip header

data = []
for post_text in posts:
    try:
        # Extract fields using regex patterns
        post_match = re.search(r"^(.*?)\n", post_text)
        centre_match = re.search(r"CENTRE\s*:\s*(.*)", post_text)
        salary_match = re.search(r"SALARY\s*:\s*(.*)", post_text)
        requirements_match = re.search(r"REQUIREMENTS\s*:\s*(.*?)(?:DUTIES\s*:)", post_text, re.DOTALL)
        duties_match = re.search(r"DUTIES\s*:\s*(.*?)(?:ENQUIRIES\s*:)", post_text, re.DOTALL)
        enquiries_match = re.search(r"ENQUIRIES\s*:\s*(.*)", post_text)
        # The closing date regex was applied to the entire text, not the individual post text,
        # which might lead to incorrect results if there are multiple closing dates.
        # Assuming the closing date is at the end of the entire document or consistent across posts.
        # If it's per post, this regex needs adjustment to be within the post_text.
        closing_match = re.search(r"CLOSING DATE\s*:\s*(.*)", text)


        post = post_match.group(1).strip() if post_match else ""
        centre = centre_match.group(1).strip() if centre_match else ""
        salary = salary_match.group(1).strip() if salary_match else ""
        requirements = requirements_match.group(1).strip().replace("\n", " ") if requirements_match else ""
        duties = duties_match.group(1).strip().replace("\n", " ") if duties_match else ""
        enquiries = enquiries_match.group(1).strip() if enquiries_match else ""
        closing_date = closing_match.group(1).strip() if closing_match else ""


        data.append({
            "Post": post,
            "Centre": centre,
            "Salary": salary,
            "Requirements": requirements,
            "Duties": duties,
            "Enquiries": enquiries,
            "Closing Date": closing_date
        })
    except Exception as e:
        print(f"Error processing a post: {e}")
        continue
# Save to CSV
df = pd.DataFrame(data)
df

,Post,Centre,Salary,Requirements,Duties,Enquiries,Closing Date
0,DIRECTOR: ATMOSPHERIC POLICY REGULATIONS AND P...,Pretoria,"R1 216 824 per annum, all-inclusive remunerati...",An undergraduate qualification in Natural or P...,Manage the identification and development of n...,Dr P Gwaze Tel No: (012) 399 9362,10 March 2025
1,PROJECT MANAGER: GLOBAL ENVIRONMENT FACILITY 7...,Pretoria,R1 003 890 per annum,Degree/National Diploma (NQF6) in project mana...,Provide strategic leadership to the project te...,Mr S Malete Tel No: (012) 399 9511,10 March 2025
2,SENIOR EMPLOYEE HEALTH AND WELLNESS PRACTITION...,Bisho,R376 413 per annum,Degree in Social Work or Honors Degree (NQF8) ...,Implementation of healthy lifestyle promotion ...,Ms N Khumalo Tel No: (012) 399 8528,10 March 2025
3,SENIOR SUPPLY CHAIN MANAGEMENT CLERK: DEMAND AND,Pretoria,R255 450 per annum (Level 06),An appropriate National Diploma (NQF Level 6) ...,The successful candidate will be responsible t...,Ms. Mpho Ramashi Tel No: (012) 473 0194,10 March 2025
4,SENIOR SUPPLY CHAIN MANAGEMENT CLERK: LOGISTIC...,Pretoria,R255 450 per annum (Level 06),An appropriate National Diploma (NQF Level 6) ...,The successful candidate will be an entry poin...,Ms. Mary-Jane Rabodiba Tel No: (012) 473 0172,10 March 2025
...,...,...,...,...,...,...,...
308,MEDICAL OFFICER GRADE 1 TO 3 (20 SESSIONS P/WE...,George Sub District,Grade 1: R457 per hour,Minimum educational qualification: Appropriate...,Provide quality outpatient care to patients in...,Ms S Pienaar Tel No: (044) 803-2703,10 March 2025
309,DENTIST GRADE 1 TO 3 (20 SESSIONS PER WEEK) (X...,NHI Project Garden Route District (Various Ins...,Grade 1: R444 per hour,Minimum educational qualification: Appropriate...,Provide clinical primary and secondary dental ...,Ms S Pienaar Tel No: (044) 803 2703,10 March 2025
310,PHARMACIST GRADE 1 TO 3 (20 SESSIONS PER WEEK)...,"George Sub District, Mossel Bay Sub District (...",,Minimum educational qualification: A qualifica...,Pharmaceutical service delivery including impr...,Ms S Pienaar Tel No: (044) 803 2703,10 March 2025
311,PHYSIOTHERAPIST GRADE 1 TO 3 (20 SESSIONS PER ...,George Sub-district,Grade 1: R248 per hour,Minimum educational qualification: Appropriate...,Provide clinical physiotherapy service deliver...,Dr TS Ackerman Tel No: (044) 814 1124,10 March 2025


In [ ]:
print("this government circular has",len(df),"positions")

this government circular has 313 positions


In [ ]:
df["Salary"]

,Salary
0,"R1 216 824 per annum, all-inclusive remunerati..."
1,R1 003 890 per annum
2,R376 413 per annum
3,R255 450 per annum (Level 06)
4,R255 450 per annum (Level 06)
...,...
308,Grade 1: R457 per hour
309,Grade 1: R444 per hour
310,
311,Grade 1: R248 per hour


In [ ]:

df['Amount'] = df['Salary'].str.extract(r'(R[\d\s]+)')
df['Period'] = df['Salary'].str.extract(r'per\s+(\w+)')

In [ ]:


Yes pdf

,Post,Centre,Salary,Requirements,Duties,Enquiries,Closing Date,Amount,Period
0,DIRECTOR: ATMOSPHERIC POLICY REGULATIONS AND P...,Pretoria,"R1 216 824 per annum, all-inclusive remunerati...",An undergraduate qualification in Natural or P...,Manage the identification and development of n...,Dr P Gwaze Tel No: (012) 399 9362,10 March 2025,R1 216 824,annum
1,PROJECT MANAGER: GLOBAL ENVIRONMENT FACILITY 7...,Pretoria,R1 003 890 per annum,Degree/National Diploma (NQF6) in project mana...,Provide strategic leadership to the project te...,Mr S Malete Tel No: (012) 399 9511,10 March 2025,R1 003 890,annum
2,SENIOR EMPLOYEE HEALTH AND WELLNESS PRACTITION...,Bisho,R376 413 per annum,Degree in Social Work or Honors Degree (NQF8) ...,Implementation of healthy lifestyle promotion ...,Ms N Khumalo Tel No: (012) 399 8528,10 March 2025,R376 413,annum
3,SENIOR SUPPLY CHAIN MANAGEMENT CLERK: DEMAND AND,Pretoria,R255 450 per annum (Level 06),An appropriate National Diploma (NQF Level 6) ...,The successful candidate will be responsible t...,Ms. Mpho Ramashi Tel No: (012) 473 0194,10 March 2025,R255 450,annum
4,SENIOR SUPPLY CHAIN MANAGEMENT CLERK: LOGISTIC...,Pretoria,R255 450 per annum (Level 06),An appropriate National Diploma (NQF Level 6) ...,The successful candidate will be an entry poin...,Ms. Mary-Jane Rabodiba Tel No: (012) 473 0172,10 March 2025,R255 450,annum
...,...,...,...,...,...,...,...,...,...
308,MEDICAL OFFICER GRADE 1 TO 3 (20 SESSIONS P/WE...,George Sub District,Grade 1: R457 per hour,Minimum educational qualification: Appropriate...,Provide quality outpatient care to patients in...,Ms S Pienaar Tel No: (044) 803-2703,10 March 2025,R457,hour
309,DENTIST GRADE 1 TO 3 (20 SESSIONS PER WEEK) (X...,NHI Project Garden Route District (Various Ins...,Grade 1: R444 per hour,Minimum educational qualification: Appropriate...,Provide clinical primary and secondary dental ...,Ms S Pienaar Tel No: (044) 803 2703,10 March 2025,R444,hour
310,PHARMACIST GRADE 1 TO 3 (20 SESSIONS PER WEEK)...,"George Sub District, Mossel Bay Sub District (...",,Minimum educational qualification: A qualifica...,Pharmaceutical service delivery including impr...,Ms S Pienaar Tel No: (044) 803 2703,10 March 2025,NaN,NaN
311,PHYSIOTHERAPIST GRADE 1 TO 3 (20 SESSIONS PER ...,George Sub-district,Grade 1: R248 per hour,Minimum educational qualification: Appropriate...,Provide clinical physiotherapy service deliver...,Dr TS Ackerman Tel No: (044) 814 1124,10 March 2025,R248,hour


In [ ]:
df['Amount']

,count
Period,
annum,306
hour,4
